In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# Carregar o dataset
df = pd.read_csv("data.csv", sep=',')
df = df.drop(columns=["Code Gdo"]).dropna()

In [ ]:
# Função para remover outliers numéricos usando IQR (excluindo a variável target "Durée")
def remove_outliers_iqr(dataframe, columns):
    df_no_outliers = dataframe.copy()
    for col in columns:
        if pd.api.types.is_numeric_dtype(df_no_outliers[col]) and col != "Durée":
            Q1 = df_no_outliers[col].quantile(0.25)
            Q3 = df_no_outliers[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            df_no_outliers = df_no_outliers[(df_no_outliers[col] >= lower) & (df_no_outliers[col] <= upper)]
    return df_no_outliers

# Remover outliers (exceto na coluna "Durée")
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
df = remove_outliers_iqr(df, numeric_cols)

In [ ]:
# Converter o target para categorias
def categorize_duration(duration):
    if duration <= 90:
        return 0
    elif duration <= 120:
        return 1
    else:
        return 2

df["Durée_cat"] = df["Durée"].apply(categorize_duration)
df = df.drop(columns=["Durée"])

In [ ]:
# Separar X e y
X = df.drop(columns=["Durée_cat"])
y = df["Durée_cat"]

# Separar em treino, validação e teste (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Identificar colunas numéricas e categóricas
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# Pré-processamento
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])


In [ ]:
# Modelos e grids de parâmetros
model_grids = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000, class_weight='balanced'),
        "params": {
            'model__C': [0.01, 0.1, 1, 10]
        }
    },
    "RidgeClassifier": {
        "model": RidgeClassifier(class_weight='balanced'),
        "params": {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=0, class_weight='balanced'),
        "params": {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 10, 20]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=0),
        "params": {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [3, 5]
        }
    }
}

best_models = {}

In [ ]:
# GridSearch para cada modelo
for name, config in model_grids.items():
    print(f"\n🔍 Treinando modelo: {name}")

    # Pipeline com SMOTE
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', config["model"])
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid=config["params"],
        scoring='balanced_accuracy',
        cv=3,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    print(f"Melhores parâmetros: {grid.best_params_}")
    print(f"Melhor balanced accuracy (validação): {grid.best_score_:.3f}")

    best_models[name] = grid.best_estimator_

# Avaliação final no conjunto de teste
print("\n🎯 Avaliação no conjunto de teste:")

In [ ]:
for name, model in best_models.items():
    y_pred = model.predict(X_test)

    print(f"\n📌 Modelo: {name}")
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
    print(f"F1 Score (macro): {f1_score(y_test, y_pred, average='macro'):.3f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.3f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.3f}")
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred, zero_division=0))